In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR.parent / "data" / "processed"

customers = pd.read_csv(PROCESSED_DIR / "customers_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv")
transaction_items = pd.read_csv(PROCESSED_DIR / "transaction_items_clean.csv")
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv")
support = pd.read_csv(PROCESSED_DIR / "customer_support_clean.csv")
marketing = pd.read_csv(PROCESSED_DIR / "marketing_touchpoints_clean.csv")
stores = pd.read_csv(PROCESSED_DIR / "stores_clean.csv")

In [17]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(" ", "_", regex=False)
    )
    return df

customers = normalize_columns(customers)
transactions = normalize_columns(transactions)
transaction_items = normalize_columns(transaction_items)
products = normalize_columns(products)
support = normalize_columns(support)
marketing = normalize_columns(marketing)
stores = normalize_columns(stores)

for name, df in {
    "customers": customers,
    "transactions": transactions,
    "transaction_items": transaction_items,
    "products": products,
    "support": support,
    "marketing": marketing,
    "stores": stores
}.items():
    print(name, df.columns.tolist())

customers ['customer_id', 'gender', 'birth_year', 'city', 'registration_date', 'acquisition_channel', 'birth_year_missing_flag', 'registration_date_clean']
transactions ['transaction_id', 'customer_id', 'transaction_date', 'store_id', 'payment_method', 'discount_applied', 'transaction_date_clean']
transaction_items ['transaction_id', 'product_id', 'quantity', 'unit_price']
products ['product_id', 'product_name', 'category', 'sub_category', 'brand', 'base_price']
support ['ticket_id', 'customer_id', 'created_date', 'channel', 'issue_type', 'resolution_status', 'satisfaction_score', 'channel_clean', 'created_date_clean', 'pending_with_score_flag']
marketing ['campaign_id', 'customer_id', 'sent_date', 'channel', 'campaign_type', 'is_opened', 'is_clicked', 'offer_value', 'is_opened_clean', 'is_clicked_clean', 'channel_clean', 'sent_date_clean']
stores ['store_id', 'store_name', 'city', 'store_type', 'opened_date', 'opened_date_clean']


In [18]:
date_cols = {
    "customers": ["registration_date", "registration_date_clean"],
    "transactions": ["transaction_date", "transaction_date_clean"],
    "support": ["created_date", "created_date_clean"],
    "marketing": ["sent_date", "sent_date_clean"],
    "stores": ["opened_date", "opened_date_clean"]
}

for col in date_cols["transactions"]:
    if col in transactions.columns:
        transactions[col] = pd.to_datetime(transactions[col], errors="coerce")

for col in date_cols["customers"]:
    if col in customers.columns:
        customers[col] = pd.to_datetime(customers[col], errors="coerce")

for col in date_cols["support"]:
    if col in support.columns:
        support[col] = pd.to_datetime(support[col], errors="coerce")

for col in date_cols["marketing"]:
    if col in marketing.columns:
        marketing[col] = pd.to_datetime(marketing[col], errors="coerce")

for col in date_cols["stores"]:
    if col in stores.columns:
        stores[col] = pd.to_datetime(stores[col], errors="coerce")

In [19]:
transaction_items["quantity"] = pd.to_numeric(transaction_items["quantity"], errors="coerce")
transaction_items["unit_price"] = pd.to_numeric(transaction_items["unit_price"], errors="coerce")

transaction_items["item_amount"] = transaction_items["quantity"] * transaction_items["unit_price"]

tx_amount = (
    transaction_items
    .groupby("transaction_id", as_index=False)
    .agg(
        gross_amount=("item_amount", "sum"),
        total_items=("quantity", "sum"),
        unique_products=("product_id", "nunique")
    )
)

tx_amount.head()

,transaction_id,gross_amount,total_items,unique_products
0,T000001,1710000.0,5.0,1
1,T000002,149082000.0,14.0,2
2,T000003,4836000.0,2.0,1
3,T000004,5687000.0,13.0,4
4,T000005,71016000.0,8.0,4


In [20]:
transactions["discount_applied"] = pd.to_numeric(transactions["discount_applied"], errors="coerce")
transactions["discount_applied"] = transactions["discount_applied"].fillna(0)
transactions["discount_applied"] = transactions["discount_applied"].clip(0, 1)

transactions_enriched = transactions.merge(
    tx_amount,
    on="transaction_id",
    how="left"
)

transactions_enriched["gross_amount"] = transactions_enriched["gross_amount"].fillna(0)
transactions_enriched["total_items"] = transactions_enriched["total_items"].fillna(0)
transactions_enriched["unique_products"] = transactions_enriched["unique_products"].fillna(0)

transactions_enriched["total_amount"] = (
    transactions_enriched["gross_amount"] * (1 - transactions_enriched["discount_applied"])
)

transactions_enriched[[
    "transaction_id", "customer_id", "gross_amount", "discount_applied", "total_amount"
]].head()

,transaction_id,customer_id,gross_amount,discount_applied,total_amount
0,T000001,C00301,1710000.0,0.20,1368000.0
1,T000002,C00109,149082000.0,0.00,149082000.0
2,T000003,C00008,4836000.0,0.00,4836000.0
3,T000004,C00754,5687000.0,0.15,4833950.0
4,T000005,C00268,71016000.0,0.05,67465200.0


In [21]:
transaction_fact = transactions_enriched.merge(
    stores,
    on="store_id",
    how="left",
    suffixes=("", "_store")
)

transaction_fact.head()

,transaction_id,customer_id,transaction_date,store_id,payment_method,discount_applied,transaction_date_clean,gross_amount,total_items,unique_products,total_amount,store_name,city,store_type,opened_date,opened_date_clean
0,T000001,C00301,2025-01-01,S018,Bank Transfer,0.20,2025-01-01,1710000.0,5.0,1,1368000.0,Store_B8,Bắc Ninh,offline,2020-06-10,2020-06-10
1,T000002,C00109,2026-01-07,S014,VNPay,0.00,2026-01-07,149082000.0,14.0,2,149082000.0,Store_B4,Nha Trang,online,2020-07-08,2020-07-08
2,T000003,C00008,2024-11-29,S007,ZaloPay,0.00,2024-11-29,4836000.0,2.0,1,4836000.0,Store_A7,Hà Nội,online,2020-03-06,2020-03-06
3,T000004,C00754,2025-02-26,S010,Cash,0.15,2025-02-26,5687000.0,13.0,4,4833950.0,Store_B0,Huế,offline,2021-03-27,2021-03-27
4,T000005,C00268,2025-06-29,S023,ZaloPay,0.05,2025-06-29,71016000.0,8.0,4,67465200.0,Store_C3,Nha Trang,offline,2020-07-25,2020-07-25


In [22]:
item_detail = transaction_items.merge(
    products,
    on="product_id",
    how="left",
    suffixes=("", "_product")
)

item_detail = item_detail.merge(
    transactions[["transaction_id", "customer_id", "store_id", "transaction_date", "transaction_date_clean", "discount_applied"]],
    on="transaction_id",
    how="left"
)

item_detail.head()

,transaction_id,product_id,quantity,unit_price,item_amount,product_name,category,sub_category,brand,base_price,customer_id,store_id,transaction_date,transaction_date_clean,discount_applied
0,T000001,P0107,5.0,342000.0,1710000.0,Thực phẩm khô_107,Thực phẩm,Thực phẩm khô,TH True,342000.0,C00301,S018,2025-01-01,2025-01-01,0.20
1,T000002,P0028,4.0,4673000.0,18692000.0,Tablet_28,Điện tử,Tablet,LG,4673000.0,C00109,S014,2026-01-07,2026-01-07,0.00
2,T000002,P0007,10.0,13039000.0,130390000.0,Laptop_7,Điện tử,Laptop,Samsung,13039000.0,C00109,S014,2026-01-07,2026-01-07,0.00
3,T000003,P0065,2.0,2418000.0,4836000.0,Nhà bếp_65,Gia dụng,Nhà bếp,Generic,2418000.0,C00008,S007,2024-11-29,2024-11-29,0.00
4,T000004,P0092,2.0,999000.0,1998000.0,Trang trí_92,Gia dụng,Trang trí,Local Brand,999000.0,C00754,S010,2025-02-26,2025-02-26,0.15


In [23]:
if "created_date_clean" in support.columns:
    support_date_col = "created_date_clean"
elif "created_date" in support.columns:
    support_date_col = "created_date"
else:
    support_date_col = None

support["is_resolved"] = support["resolution_status"].eq("Resolved").astype(int)
support["is_pending"] = support["resolution_status"].eq("Pending").astype(int)

support_agg = support.groupby("customer_id", as_index=False).agg(
    support_ticket_count=("ticket_id", "count") if "ticket_id" in support.columns else ("customer_id", "size"),
    resolved_rate=("is_resolved", "mean"),
    pending_ticket_count=("is_pending", "sum"),
    avg_satisfaction_score=("satisfaction_score", "mean")
)

if support_date_col is not None:
    last_support = (
        support.groupby("customer_id", as_index=False)[support_date_col]
        .max()
        .rename(columns={support_date_col: "last_support_date"})
    )
    support_agg = support_agg.merge(last_support, on="customer_id", how="left")

support_agg.head()

,customer_id,support_ticket_count,resolved_rate,pending_ticket_count,avg_satisfaction_score,last_support_date
0,C00003,1,1.000000,0,5.000000,2023-10-02
1,C00004,3,0.666667,1,5.000000,2025-06-12
2,C00005,3,1.000000,0,3.333333,2024-06-03
3,C00006,4,0.750000,1,4.666667,2024-07-29
4,C00009,1,0.000000,0,4.000000,2025-06-22


In [24]:
# fallback tên cột
open_col = "is_opened_clean" if "is_opened_clean" in marketing.columns else "is_opened"
click_col = "is_clicked_clean" if "is_clicked_clean" in marketing.columns else "is_clicked"
date_col = "sent_date_clean" if "sent_date_clean" in marketing.columns else "sent_date"

marketing[open_col] = pd.to_numeric(marketing[open_col], errors="coerce").fillna(0)
marketing[click_col] = pd.to_numeric(marketing[click_col], errors="coerce").fillna(0)

if "offer_value" in marketing.columns:
    marketing["offer_value"] = pd.to_numeric(marketing["offer_value"], errors="coerce")
    marketing["offer_value"] = marketing["offer_value"].fillna(0).clip(lower=0)
else:
    marketing["offer_value"] = 0

marketing_agg = marketing.groupby("customer_id", as_index=False).agg(
    campaign_count=("campaign_id", "count") if "campaign_id" in marketing.columns else ("customer_id", "size"),
    open_rate=(open_col, "mean"),
    click_rate=(click_col, "mean"),
    total_offer_value=("offer_value", "sum")
)

if date_col in marketing.columns:
    last_campaign = (
        marketing.groupby("customer_id", as_index=False)[date_col]
        .max()
        .rename(columns={date_col: "last_campaign_date"})
    )
    marketing_agg = marketing_agg.merge(last_campaign, on="customer_id", how="left")

marketing_agg.head()

,customer_id,campaign_count,open_rate,click_rate,total_offer_value,last_campaign_date
0,C00001,2,1.000000,0.000000,40000.0,2025-04-12
1,C00002,3,0.333333,0.000000,70000.0,2025-12-13
2,C00003,6,0.333333,0.166667,125000.0,2026-02-16
3,C00004,5,0.400000,0.200000,105000.0,2026-03-15
4,C00005,2,0.000000,0.000000,5000.0,2025-10-02


In [25]:
tx_date_col = "transaction_date_clean" if "transaction_date_clean" in transaction_fact.columns else "transaction_date"

customer_tx_agg = transaction_fact.groupby("customer_id", as_index=False).agg(
    transaction_count=("transaction_id", "count"),
    total_revenue=("total_amount", "sum"),
    avg_order_value=("total_amount", "mean"),
    total_discount_rate=("discount_applied", "sum"),
    avg_discount_rate=("discount_applied", "mean"),
    total_items_bought=("total_items", "sum"),
    avg_items_per_order=("total_items", "mean"),
    n_unique_stores=("store_id", "nunique")
)

if tx_date_col in transaction_fact.columns:
    tx_dates = transaction_fact.groupby("customer_id", as_index=False).agg(
        first_purchase_date=(tx_date_col, "min"),
        last_purchase_date=(tx_date_col, "max")
    )
    customer_tx_agg = customer_tx_agg.merge(tx_dates, on="customer_id", how="left")

customer_tx_agg.head()

,customer_id,transaction_count,total_revenue,avg_order_value,total_discount_rate,avg_discount_rate,total_items_bought,avg_items_per_order,n_unique_stores,first_purchase_date,last_purchase_date
0,C00001,10,81962050.0,8.196205e+06,1.05,0.105000,43.0,4.300000,6,2024-06-04,2026-02-01
1,C00002,12,230204400.0,1.918370e+07,0.90,0.075000,82.0,6.833333,9,2025-04-17,2026-03-10
2,C00003,13,52317400.0,4.024415e+06,1.15,0.088462,58.0,4.461538,11,2023-09-09,2026-03-01
3,C00004,6,131854050.0,2.197568e+07,0.40,0.066667,44.0,7.333333,6,2025-05-10,2026-03-21
4,C00005,10,99898600.0,9.989860e+06,1.25,0.125000,67.0,6.700000,9,2024-04-23,2026-02-13


In [26]:
customer_360 = customers.merge(
    customer_tx_agg, on="customer_id", how="left"
).merge(
    support_agg, on="customer_id", how="left"
).merge(
    marketing_agg, on="customer_id", how="left"
)

customer_360.head()

,customer_id,gender,birth_year,city,registration_date,acquisition_channel,birth_year_missing_flag,registration_date_clean,transaction_count,total_revenue,...,support_ticket_count,resolved_rate,pending_ticket_count,avg_satisfaction_score,last_support_date,campaign_count,open_rate,click_rate,total_offer_value,last_campaign_date
0,C00001,F,1979.0,Hải Phòng,2024-05-24,Facebook,0,2024-05-24,10.0,81962050.0,...,NaN,NaN,NaN,NaN,NaT,2.0,1.000000,0.000000,40000.0,2025-04-12
1,C00002,M,1990.0,Biên Hòa,2025-04-06,Referral,0,2025-04-06,12.0,230204400.0,...,NaN,NaN,NaN,NaN,NaT,3.0,0.333333,0.000000,70000.0,2025-12-13
2,C00003,F,1987.0,NaN,2023-08-11,Shopee,0,2023-08-11,13.0,52317400.0,...,1.0,1.000000,0.0,5.000000,2023-10-02,6.0,0.333333,0.166667,125000.0,2026-02-16
3,C00004,F,1966.0,TP.HCM,2025-04-26,Referral,0,2025-04-26,6.0,131854050.0,...,3.0,0.666667,1.0,5.000000,2025-06-12,5.0,0.400000,0.200000,105000.0,2026-03-15
4,C00005,F,1967.0,TP.HCM,NaT,Shopee,0,NaT,10.0,99898600.0,...,3.0,1.000000,0.0,3.333333,2024-06-03,2.0,0.000000,0.000000,5000.0,2025-10-02


In [27]:
fill_zero_cols = [
    "transaction_count", "total_revenue", "avg_order_value", "total_discount_rate",
    "avg_discount_rate", "total_items_bought", "avg_items_per_order", "n_unique_stores",
    "support_ticket_count", "resolved_rate", "pending_ticket_count", "avg_satisfaction_score",
    "campaign_count", "open_rate", "click_rate", "total_offer_value"
]

for col in fill_zero_cols:
    if col in customer_360.columns:
        customer_360[col] = customer_360[col].fillna(0)

In [28]:
transaction_fact.to_csv(PROCESSED_DIR / "transaction_fact.csv", index=False)
item_detail.to_csv(PROCESSED_DIR / "item_detail.csv", index=False)
customer_360.to_csv(PROCESSED_DIR / "customer_360.csv", index=False)

print("Saved:")
print(PROCESSED_DIR / "transaction_fact.csv")
print(PROCESSED_DIR / "item_detail.csv")
print(PROCESSED_DIR / "customer_360.csv")

Saved:
/Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed/transaction_fact.csv
/Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed/item_detail.csv
/Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed/customer_360.csv


### Integration summary

- `total_amount` was not available in the raw transactions table, so it was derived from
  `transaction_items` as the sum of `quantity * unit_price` per transaction, then adjusted
  using `discount_applied`.
- A transaction-level fact table was built by joining cleaned transactions with aggregated
  item amounts and store information.
- Customer-level support and marketing summaries were aggregated separately and then merged
  back to the customer table using LEFT JOIN to preserve customers with no transactions,
  no support tickets, or no campaign exposure.
- The final customer-level integrated table (`customer_360`) will be used as the base input
  for feature engineering and target-label construction.

In [29]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RAW exists:", RAW_DIR.exists())
print("PROCESSED exists:", PROCESSED_DIR.exists())

customers = pd.read_csv(PROCESSED_DIR / "customers_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv")
transaction_items = pd.read_csv(PROCESSED_DIR / "transaction_items_clean.csv")
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv")
stores = pd.read_csv(PROCESSED_DIR / "stores_clean.csv")
customer_support = pd.read_csv(PROCESSED_DIR / "customer_support_clean.csv")
marketing_touchpoints = pd.read_csv(PROCESSED_DIR / "marketing_touchpoints_clean.csv")

transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")

item_amount = (
    transaction_items.assign(item_amount=transaction_items["quantity"] * transaction_items["unit_price"])
    .groupby("transaction_id", as_index=False)["item_amount"]
    .sum()
    .rename(columns={"item_amount": "gross_amount"})
)

transactions_enriched = transactions.merge(item_amount, on="transaction_id", how="left")
transactions_enriched["gross_amount"] = transactions_enriched["gross_amount"].fillna(0)

transactions_enriched["discount_applied"] = pd.to_numeric(
    transactions_enriched["discount_applied"], errors="coerce"
).fillna(0)

transactions_enriched["total_amount"] = (
    transactions_enriched["gross_amount"] * (1 - transactions_enriched["discount_applied"])
).clip(lower=0)

transactions_enriched.head()

PROJECT_ROOT: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project
RAW_DIR: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/raw
PROCESSED_DIR: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed
RAW exists: True
PROCESSED exists: True


,transaction_id,customer_id,transaction_date,store_id,payment_method,discount_applied,transaction_date_clean,gross_amount,total_amount
0,T000001,C00301,2025-01-01,S018,Bank Transfer,0.20,2025-01-01,1710000.0,1368000.0
1,T000002,C00109,2026-01-07,S014,VNPay,0.00,2026-01-07,149082000.0,149082000.0
2,T000003,C00008,2024-11-29,S007,ZaloPay,0.00,2024-11-29,4836000.0,4836000.0
3,T000004,C00754,2025-02-26,S010,Cash,0.15,2025-02-26,5687000.0,4833950.0
4,T000005,C00268,2025-06-29,S023,ZaloPay,0.05,2025-06-29,71016000.0,67465200.0


In [30]:
transactions_enriched.to_csv(PROCESSED_DIR / "transactions_enriched.csv", index=False)
print("Saved:", PROCESSED_DIR / "transactions_enriched.csv")

Saved: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed/transactions_enriched.csv


## Chiến lược JOIN

| Join | Bảng | Lý do |
|---|---|---|
| **LEFT JOIN** | `customers` ← `transactions` | Giữ khách chưa có giao dịch nào (churn analysis) |
| **LEFT JOIN** | `transactions` ← `stores` | Giữ giao dịch nếu store bị thiếu dữ liệu |
| **INNER JOIN** | `transaction_items` ↔ `transactions` | Item phải có transaction tương ứng mới hợp lệ |
| **INNER JOIN** | `transaction_items` ↔ `products` | Item phải có product info để tính revenue |
| **LEFT JOIN** | `customers` ← `support_agg` | Khách chưa gửi ticket → NaN (không loại bỏ) |
| **LEFT JOIN** | `customers` ← `marketing_agg` | Khách chưa nhận campaign → NaN (không loại bỏ) |

> Nguyên tắc: dimension table là LEFT, fact table là INNER.

In [32]:
# Xem tất cả biến dạng DataFrame đang có trong namespace
import pandas as pd
df_vars = {k: v.shape for k, v in globals().items() if isinstance(v, pd.DataFrame)}
print(df_vars)

{'_': (5, 9), '__': (5, 28), '___': (5, 11), 'customers': (800, 8), 'transactions': (3500, 7), 'transaction_items': (7740, 4), 'products': (150, 6), 'support': (700, 12), 'marketing': (1800, 12), 'stores': (25, 6), 'df': (25, 6), 'tx_amount': (3500, 4), '_4': (5, 4), 'transactions_enriched': (3500, 9), '_5': (5, 5), 'transaction_fact': (3500, 16), '_6': (5, 16), 'item_detail': (7740, 15), '_7': (5, 15), 'support_agg': (448, 6), 'last_support': (448, 2), '_8': (5, 6), 'marketing_agg': (711, 6), 'last_campaign': (711, 2), '_9': (5, 6), 'customer_tx_agg': (751, 11), 'tx_dates': (751, 3), '_10': (5, 11), 'customer_360': (800, 28), '_11': (5, 28), 'customer_support': (700, 10), 'marketing_touchpoints': (1800, 12), 'item_amount': (3500, 2), '_14': (5, 9), '_19': (5, 4), '_20': (5, 5), '_21': (5, 16), '_22': (5, 15), '_23': (5, 6), '_24': (5, 6), '_25': (5, 11), '_26': (5, 28), '_29': (5, 9)}


In [33]:
# Kiểm tra: customer_360 phải có đúng số lượng customers
n_customers = len(customers)
assert len(customer_360) == n_customers, \
    f"Row count mismatch! Expected {n_customers}, got {len(customer_360)}"
print(f"✅ customer_360: {len(customer_360)} rows (= {n_customers} customers)")

# Kiểm tra: transactions không mất khi JOIN items
n_trans = transactions['transaction_id'].nunique()
n_trans_in_items = transaction_items['transaction_id'].nunique()
print(f"✅ Transactions có items: {n_trans_in_items}/{n_trans} ({n_trans_in_items/n_trans*100:.1f}%)")

# Kiểm tra: không có customer_id bị duplicate trong customer_360
dup_cust = customer_360['customer_id'].duplicated().sum()
assert dup_cust == 0, f"Có {dup_cust} customer_id bị duplicate!"
print(f"✅ Không có duplicate customer_id trong customer_360")

# Kiểm tra: khách không có transaction vẫn tồn tại (LEFT JOIN đúng)
no_tx = customer_360['transaction_count'].isna().sum()
print(f"ℹ️  Khách không có giao dịch: {no_tx} (LEFT JOIN giữ lại đúng)")

✅ customer_360: 800 rows (= 800 customers)
✅ Transactions có items: 3500/3500 (100.0%)
✅ Không có duplicate customer_id trong customer_360
ℹ️  Khách không có giao dịch: 0 (LEFT JOIN giữ lại đúng)
